# GAE_HEP on Kaggle GPU: hyper-parameter study of the energy-flow graph VAE

Permutation-invariant graph autoencoders for unsupervised anomaly detection on jets of the LHC Olympics 2020 R&D dataset ([github.com/raktim711/GAE_HEP](https://github.com/raktim711/GAE_HEP); results and discussion in `RESULTS.md`).

This notebook runs the GPU part of the hyper-parameter study (`gaehep sweep`), centred on what the CPU sweep found (narrow kernel scales, short training, small latents):

1. data: download (Zenodo record 6466204, about 3.2 GB), clustering, tables, splits, single-observable baselines;
2. **grid sweep** under the development protocol (validation splits only): latent size (3, 4, 6, 8) x VAE beta (1e-3, 1e-2) x kernel scales of the energy MMD (0.02-0.05, 0.02-0.1, 0.02-0.4) x training epochs (2, 3, 5): 72 points in random order, one seed each; every point is scored with its reconstruction score and with the rank ensemble of reconstruction and KL scores (the score kind is one more hyper-parameter);
3. **seed confirmation** of the 3 best points (3 seeds), then the best (point, score) pair with **5 seeds scored on the test splits** (2-prong and the never-tuned 3-prong signal) and a seed ensemble;
4. mass decorrelation, bump hunt and the weak-supervision study for the best configuration;
5. scale variants of the best configuration (100 constituents, 128-wide or deeper encoder, 64 decoder points, 300k jets per epoch), two seeds each;
6. reports and a zip of all results (`/kaggle/working/gaehep_results.zip`, refreshed after every phase, so a time-out still leaves the finished part).

> **Before you run:** in the Kaggle session options (right-hand panel) switch **Internet on** (needed for `git clone`, `pip install` and the Zenodo download) and select a **GPU accelerator**. Run it with *Save Version -> Save & Run All (Commit)*: the whole study takes about 9-10 h.

In [ ]:
!nvidia-smi

import os
import time

# notebook start time, for the time guards of the study cell (Kaggle stops a session after 12 h)
os.environ.setdefault("GAEHEP_NB_T0", str(time.time()))

REPO_URL = "https://github.com/raktim711/GAE_HEP.git"
REPO_DIR = "/kaggle/working/GAE_HEP"

# clone once; skipped when the repo is already there (e.g. after a kernel restart)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .

import torch

print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU visible - choose a GPU accelerator in the session options.")

In [ ]:
import os
import shutil

# Results go to /kaggle/working (saved with the notebook version).  The ~6 GB of data go to /tmp when there is
# room (not saved as notebook output, which is limited to 20 GB), otherwise to /kaggle/working/data.
DATA_DIR = "/tmp/gaehep_data"
if not (os.path.isdir(DATA_DIR) or shutil.disk_usage("/tmp").free > 20e9):
    DATA_DIR = "/kaggle/working/data"
os.environ["GAEHEP_DATA"] = DATA_DIR
os.environ["GAEHEP_RESULTS"] = "/kaggle/working/results"
print("data ->", DATA_DIR, "| free disk in /tmp:", round(shutil.disk_usage("/tmp").free / 1e9), "GB")

# data pipeline; the shell commands inherit os.environ.
# `download` is md5-verified and resumable: re-run this cell if it gets interrupted.
!gaehep download
!gaehep preprocess --sample rnd --workers 4
!gaehep preprocess --sample qqq --workers 4
!gaehep tables --sample rnd
!gaehep tables --sample qqq
!gaehep splits
!gaehep baselines

## GPU study

Everything goes through the `gaehep` command line; each step is idempotent, so a re-run reuses finished runs. The sweep has a wall-clock budget (`--budget-hours`): it stops launching grid points after 60 % of it and seed runs after 85-90 %; the later phases check the time since the notebook started, so the notebook ends within Kaggle's 12 h limit.

In [ ]:
import json
import os
import shlex
import subprocess
import time
import zipfile
from pathlib import Path

REPO_DIR = "/kaggle/working/GAE_HEP"
RESULTS = Path("/kaggle/working/results")
ZIP_PATH = Path("/kaggle/working/gaehep_results.zip")

# set again so that this cell (and the next ones) also work after a kernel restart (same choice as above)
os.environ["GAEHEP_DATA"] = "/tmp/gaehep_data" if os.path.isdir("/tmp/gaehep_data") else "/kaggle/working/data"
os.environ["GAEHEP_RESULTS"] = str(RESULTS)
os.chdir(REPO_DIR)
T_START = float(os.environ.get("GAEHEP_NB_T0", time.time()))  # set in the first cell


def hours():
    """Hours since the notebook started."""
    return (time.time() - T_START) / 3600


def gaehep(*args, check=True):
    """Run `gaehep <args>`; the log streams into the cell output."""
    cmd = ["gaehep", *map(str, args)]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, check=check, cwd=REPO_DIR, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    print(f"  ({(time.time() - t0) / 60:.1f} min, notebook at {hours():.2f} h)", flush=True)
    return r.returncode


def zip_results():
    """Zip the results (no model weights / checkpoints / large score arrays) for download."""
    n = 0
    with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for f in RESULTS.rglob("*"):
            if f.is_file() and f.suffix not in {".pt", ".ckpt"} and "checkpoint" not in f.name and f.stat().st_size < 50e6:
                z.write(f, f.relative_to(RESULTS.parent))
                n += 1
    print(f"zipped {n} files -> {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)", flush=True)


GPU = ["--set", "train.device=cuda", "--set", "train.threads=4"]

In [ ]:
import copy

import yaml

# ---- phase 1-3: grid sweep -> seed confirmation -> best (configuration, score) on the test splits (5 seeds) ----
GRID = """{
  latent_dim: [3, 4, 6, 8],
  beta: [0.001, 0.01],
  model_args.sigmas: {s2: [0.02, 0.05], s3: [0.02, 0.05, 0.1], s5: [0.02, 0.05, 0.1, 0.2, 0.4]},
  train.epochs: [2, 3, 5]
}"""
gaehep("sweep", "configs/ef_vae.yaml", "--prefix", "gpu", "--device", "cuda", "--threads", "4",
       "--grid", GRID, "--score", "best",
       "--budget-hours", "6.5", "--top-k", "3", "--seeds", "3", "--final-seeds", "5")
zip_results()

best = json.loads((RESULTS / "dev" / "gpu_sweep.json").read_text())["best"]
print("best configuration:", best, flush=True)
final = "gpu_best_seeds+ens" if best["kind"] == "ens" else "gpu_best_seeds"

# ---- phase 4: post-hoc analyses of the best configuration ----
gaehep("qn", final, check=False)
gaehep("bumphunt", final, "--agg", "prod", check=False)
if hours() < 9.0:
    gaehep("weak", "gpu_best_s1", "--n-models", "5", check=False)
zip_results()

# ---- phase 5: scale variants of the best configuration (dev protocol, two seeds each) ----
best_cfg = yaml.safe_load((RESULTS / "dev" / best["name"] / "config.yaml").read_text())
ma = best_cfg["model_args"]
k16 = max(16, int(ma.get("k", 12)))
SCALE = {
    "n100": {"data.n_const": 100, "model_args.k": k16},
    "w128": {"model_args.hidden": [128] * len(ma["hidden"]), "model_args.k": k16},
    "deep": {"model_args.hidden": [*ma["hidden"], ma["hidden"][-1]]},
    "nout64": {"model_args.n_out": 64},
    "j300k": {"train.jets_per_epoch": 300_000},
}
cfg_dir = Path("/kaggle/working/scale_configs")
cfg_dir.mkdir(exist_ok=True)
for seed in (1, 2):
    for tag, changes in SCALE.items():
        if hours() > 10.0:
            break
        cfg = copy.deepcopy(best_cfg)
        for key, val in changes.items():
            node = cfg
            *parents, last = key.split(".")
            for q in parents:
                node = node.setdefault(q, {})
            node[last] = val
        cfg["train"]["seed"] = seed
        path = cfg_dir / f"{tag}_s{seed}.yaml"
        path.write_text(yaml.safe_dump(cfg, sort_keys=False))
        # `gaehep dev` resets the train section to the dev budget: pass this configuration's train settings back
        train = [x for k, v in cfg["train"].items() for x in ("--set", f"train.{k}={v}")]
        name = f"gpu_scale_{tag}" + ("" if seed == 1 else f"_s{seed}")
        gaehep("dev", path, "--name", name, *train, check=False)
        if best["kind"] == "ens" and (RESULTS / "dev" / f"{name}+kl").is_dir():
            gaehep("dev-ensemble", name, f"{name}+kl", "--out", f"{name}+ens", check=False)

gaehep("report", check=False)
gaehep("dev-report", check=False)
zip_results()

In [ ]:
import os
import zipfile
from pathlib import Path

from IPython.display import FileLink, Markdown, display

RESULTS = Path(os.environ.get("GAEHEP_RESULTS", "/kaggle/working/results"))
ZIP_PATH = Path("/kaggle/working/gaehep_results.zip")

# 1) the summary table written by `gaehep report`
summary = RESULTS / "summary.md"
if summary.is_file():
    display(Markdown(summary.read_text()))
else:
    print(f"{summary} not found - run the previous cell (it ends with `gaehep report`) first.")


# 2) zip the results for download, leaving out model weights / checkpoints
def is_checkpoint(path: Path) -> bool:
    parts = [p.lower() for p in path.relative_to(RESULTS).parts]
    return path.suffix.lower() in {".pt", ".pth", ".ckpt"} or any(
        "checkpoint" in p or p.startswith("ckpt") for p in parts
    )


added = skipped = 0
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(RESULTS.rglob("*")):
        if not path.is_file():
            continue
        if is_checkpoint(path):
            skipped += 1
            continue
        zf.write(path, arcname=str(Path("results") / path.relative_to(RESULTS)))
        added += 1

print(f"{ZIP_PATH}: {added} files, {ZIP_PATH.stat().st_size / 1e6:.1f} MB "
      f"({skipped} *.pt / checkpoint files left out)")

# download link (the file also shows up in the notebook's Output tab)
cwd = os.getcwd()
try:
    os.chdir(ZIP_PATH.parent)
    display(FileLink(ZIP_PATH.name))
finally:
    os.chdir(cwd)